# RoverLite: Edge AI-Based Terrain Classification for Mars Rover Mobility

**Deep Learning mini-project | Google Colab notebook**

This notebook builds a compact **MobileNetV2 transfer-learning model** that classifies a labelled Mars terrain image patch as **Soil, Bedrock, Sand, or Big Rock**. It evaluates held-out images, reports accuracy and macro-F1, measures CPU inference latency and saved-model size, and attempts TensorFlow Lite export.

**Dataset:** [AI4Mars on Hugging Face](https://huggingface.co/datasets/hassanjbara/AI4MARS), a convenient mirror containing rover images and pixel-level terrain masks. The original NASA AI4Mars dataset was created for terrain-aware rover research and contains image-level terrain annotations in segmentation-mask form. [NASA dataset page](https://data.nasa.gov/dataset/ai4mars-a-dataset-for-terrain-aware-autonomous-driving-on-mars).

**Important scope:** The model predicts a dominant label for a *terrain patch*, not a complete route or a certified safe/unsafe command. `SAND` and low-confidence predictions trigger an advisory caution only. This is a laptop/Colab prototype; Raspberry Pi deployment is not claimed unless you separately test on one.

### Before you run
1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** if available. CPU also works, but training will be slower.
3. Use **Runtime → Run all**. The first dataset cell streams labelled examples from Hugging Face; it may take several minutes depending on network and Colab resources.
4. Keep the output screenshots and measured values for your report/PPT. Do not write results before running the notebook.

## 1. Install / import libraries

In [ ]:
!pip -q install -U "datasets>=2.19.0" scikit-learn matplotlib pillow

import os, io, time, random, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

CLASS_NAMES = ["Soil", "Bedrock", "Sand", "Big Rock"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224           # matches the 224x224 input described in the PPT
RESIZED_SOURCE_SIZE = 256
PATCH_SIZE = RESIZED_SOURCE_SIZE // 2
MAX_LABELLED_IMAGES = 1600
MAX_PATCHES = 5000
MIN_VALID_MASK_FRACTION = 0.20
MIN_DOMINANT_FRACTION = 0.45


## 2. Load AI4Mars and turn mask-labelled images into terrain patches

AI4Mars masks use these class IDs: `0 = soil`, `1 = bedrock`, `2 = sand`, `3 = big rock`, and `255 = unlabeled/ignore`. Each original image can contain multiple terrain classes. To make a manageable classification mini-project, this cell resizes the image and mask, splits them into four quadrants, and keeps a patch only when enough valid mask pixels exist and one class is dominant. Patches from one original image remain grouped together for train/validation/test splitting.

In [ ]:
def as_pil(obj):
    """Convert a Hugging Face image feature (PIL or {bytes/path}) to PIL."""
    if isinstance(obj, Image.Image):
        return obj
    if isinstance(obj, dict):
        data = obj.get("bytes")
        path = obj.get("path")
        if data is not None:
            return Image.open(io.BytesIO(data))
        if path is not None and os.path.exists(path):
            return Image.open(path)
    if isinstance(obj, (bytes, bytearray)):
        return Image.open(io.BytesIO(obj))
    raise TypeError(f"Unsupported image object: {type(obj)}")


def get_mask_array(mask_obj):
    mask_pil = as_pil(mask_obj)
    arr = np.asarray(mask_pil)
    # AI4Mars labels are stored as grayscale/identical RGB channels in common mirrors.
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    if arr.dtype != np.uint8:
        arr = arr.astype(np.uint8)
    return arr


def extract_patches(image_obj, mask_obj):
    """Return (RGB patch, class_id) pairs using dominant valid label per quadrant."""
    image = as_pil(image_obj).convert("RGB").resize(
        (RESIZED_SOURCE_SIZE, RESIZED_SOURCE_SIZE), Image.Resampling.BILINEAR
    )
    mask = Image.fromarray(get_mask_array(mask_obj)).resize(
        (RESIZED_SOURCE_SIZE, RESIZED_SOURCE_SIZE), Image.Resampling.NEAREST
    )
    mask_arr = np.asarray(mask)
    if mask_arr.ndim == 3:
        mask_arr = mask_arr[:, :, 0]

    pairs = []
    for top in (0, PATCH_SIZE):
        for left in (0, PATCH_SIZE):
            right, bottom = left + PATCH_SIZE, top + PATCH_SIZE
            mask_patch = mask_arr[top:bottom, left:right]
            valid = np.isin(mask_patch, [0, 1, 2, 3])
            if float(valid.mean()) < MIN_VALID_MASK_FRACTION:
                continue
            valid_labels = mask_patch[valid].astype(np.int64)
            counts = np.bincount(valid_labels, minlength=NUM_CLASSES)[:NUM_CLASSES]
            total = int(counts.sum())
            if total == 0:
                continue
            label = int(np.argmax(counts))
            if counts[label] / total < MIN_DOMINANT_FRACTION:
                continue
            patch = image.crop((left, top, right, bottom)).resize(
                (IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR
            )
            pairs.append((np.asarray(patch, dtype=np.uint8), label))
    return pairs

print("Streaming dataset metadata from Hugging Face...")
stream = load_dataset("hassanjbara/AI4MARS", split="train", streaming=True)

all_images = []
all_labels = []
source_ids = []
labelled_source_count = 0
bad_rows = 0
raw_rows_seen = 0

for row in stream:
    raw_rows_seen += 1
    if not row.get("has_labels", False) or row.get("label_mask") is None:
        continue
    try:
        patches = extract_patches(row["image"], row["label_mask"])
    except Exception as exc:
        bad_rows += 1
        if bad_rows <= 3:
            print("Skipping an unreadable row:", repr(exc)[:180])
        continue

    if not patches:
        continue

    group_id = labelled_source_count
    labelled_source_count += 1
    for patch, label in patches:
        all_images.append(patch)
        all_labels.append(label)
        source_ids.append(group_id)
        if len(all_labels) >= MAX_PATCHES:
            break
    if labelled_source_count % 100 == 0:
        print(f"Labelled source images processed: {labelled_source_count}; usable patches: {len(all_labels)}")
    # Stop at either patch cap or source-image cap.
    if len(all_labels) >= MAX_PATCHES or labelled_source_count >= MAX_LABELLED_IMAGES:
        break

if len(all_labels) < 100:
    raise RuntimeError(
        "Too few labelled patches were extracted. Check internet access and the dataset fields, "
        "then inspect the first rows of the Hugging Face dataset."
    )

X = np.stack(all_images).astype(np.uint8)
y = np.asarray(all_labels, dtype=np.int64)
groups = np.asarray(source_ids, dtype=np.int32)

print("\nData extraction complete")
print("Raw rows scanned:", raw_rows_seen)
print("Labelled source images used:", labelled_source_count)
print("Usable image patches:", len(X))
print("Image tensor shape:", X.shape)
print("Class counts:", {CLASS_NAMES[i]: int((y == i).sum()) for i in range(NUM_CLASSES)})
print("Unique classes found:", [CLASS_NAMES[i] for i in np.unique(y)])
print("Unreadable rows skipped:", bad_rows)


## 3. Inspect a few training samples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.ravel(), np.linspace(0, len(X)-1, min(8, len(X)), dtype=int)):
    ax.imshow(X[idx])
    ax.set_title(CLASS_NAMES[int(y[idx])])
    ax.axis("off")
plt.suptitle("AI4Mars terrain patches and dominant mask labels")
plt.tight_layout()
plt.show()


## 4. Split by original image to reduce data leakage

In [ ]:
unique_groups = np.unique(groups)
if len(unique_groups) < 10:
    raise RuntimeError("Not enough source images were extracted for a reliable train/test split.")

# Keep every patch from the same source image in exactly one split.
train_groups, temporary_groups = train_test_split(
    unique_groups, test_size=0.30, random_state=SEED
)
val_groups, test_groups = train_test_split(
    temporary_groups, test_size=0.50, random_state=SEED
)

train_mask = np.isin(groups, train_groups)
val_mask = np.isin(groups, val_groups)
test_mask = np.isin(groups, test_groups)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train patches: {len(X_train)} from {len(train_groups)} source images")
print(f"Validation patches: {len(X_val)} from {len(val_groups)} source images")
print(f"Test patches: {len(X_test)} from {len(test_groups)} source images")
for split_name, split_labels in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(split_name, {CLASS_NAMES[i]: int((split_labels == i).sum()) for i in range(NUM_CLASSES)})

missing_train_classes = sorted(set(range(NUM_CLASSES)) - set(np.unique(y_train).tolist()))
if missing_train_classes:
    print("WARNING: These classes are absent from the training split:", [CLASS_NAMES[i] for i in missing_train_classes])
    print("The report will still include all four classes; absent classes will have zero support/recall.")


## 5. Build the lightweight MobileNetV2 model

MobileNetV2 starts with ImageNet-pretrained visual features. The backbone is frozen for the first training pass, and a small four-class softmax head learns the Mars-terrain labels. Pixel scaling is included inside the model, so inference accepts RGB pixel values from 0 to 255.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="terrain_patch_rgb")
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="scale_to_minus1_plus1")(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.20, name="dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="terrain_probabilities")(x)
model = Model(inputs, outputs, name="RoverLite_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


## 6. Train the model

In [ ]:
# Convert uint8 image batches to float32 without keeping a second full-size image array in memory.
def make_dataset(images, labels=None, batch_size=32, training=False):
    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices(images)
        ds = ds.batch(batch_size)
        ds = ds.map(lambda x: tf.cast(x, tf.float32), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((images, labels))
        if training:
            ds = ds.shuffle(min(len(images), 2048), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(batch_size)
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

present_classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(present_classes, weights)}
print("Class weights:", {CLASS_NAMES[k]: round(v, 3) for k, v in class_weight.items()})

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history.get("accuracy", []), label="Train")
axes[0].plot(history.history.get("val_accuracy", []), label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history.get("loss", []), label="Train")
axes[1].plot(history.history.get("val_loss", []), label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Evaluate on held-out test images

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probabilities = model.predict(make_dataset(X_test), verbose=0)
y_pred = np.argmax(probabilities, axis=1)
macro_f1 = f1_score(y_test, y_pred, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Macro-F1: {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d", xticks_rotation=30, colorbar=False)
plt.title("RoverLite: test confusion matrix")
plt.tight_layout()
plt.show()


## 9. Demonstrate one prediction

In [ ]:
rng = np.random.default_rng(SEED)
demo_idx = int(rng.integers(0, len(X_test)))
demo_image = X_test[demo_idx]
probs = model.predict(np.expand_dims(demo_image.astype(np.float32), axis=0), verbose=0)[0]
pred_id = int(np.argmax(probs))
confidence = float(probs[pred_id])
true_id = int(y_test[demo_idx])

if CLASS_NAMES[pred_id] == "Sand" or confidence < 0.60:
    caution_text = "CAUTION / REVIEW TERRAIN"
else:
    caution_text = "NO CAUTION TRIGGERED (prototype rule only)"

plt.figure(figsize=(5, 5))
plt.imshow(demo_image)
plt.axis("off")
plt.title(f"Predicted: {CLASS_NAMES[pred_id]} ({confidence:.1%})")
plt.show()
print("Actual patch label:", CLASS_NAMES[true_id])
print("Predicted class:", CLASS_NAMES[pred_id])
print(f"Confidence: {confidence:.4f}")
print("Advisory output:", caution_text)
print("All class probabilities:")
for name, probability in zip(CLASS_NAMES, probs):
    print(f"  {name:10s}: {probability:.4f}")
print("\nSafety note: this is a demo rule, not a rover driving command or a guarantee of terrain safety.")


## 10. Measure model size and CPU inference latency

In [ ]:
# Save the trained Keras model.
KERAS_PATH = "/content/roverlite_mobilenetv2.keras"
model.save(KERAS_PATH)
keras_size_mb = os.path.getsize(KERAS_PATH) / (1024 ** 2)

# Clone the trained model under the CPU device context to benchmark CPU inference specifically.
with tf.device("/CPU:0"):
    cpu_model = tf.keras.models.clone_model(model)
    _ = cpu_model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32), training=False)
    cpu_model.set_weights(model.get_weights())
    cpu_input = tf.convert_to_tensor(np.expand_dims(X_test[0].astype(np.float32), axis=0))
    for _ in range(10):
        _ = cpu_model(cpu_input, training=False)
    repetitions = 50
    start = time.perf_counter()
    for _ in range(repetitions):
        _ = cpu_model(cpu_input, training=False)
    cpu_latency_ms = (time.perf_counter() - start) * 1000 / repetitions

print(f"Keras model size: {keras_size_mb:.2f} MB")
print(f"Average CPU latency per patch (batch size 1): {cpu_latency_ms:.2f} ms")
print("Latency is measured on this Colab runtime; it is not a Raspberry Pi result.")


## 11. Optional: convert to TensorFlow Lite

In [ ]:
TFLITE_PATH = "/content/roverlite_mobilenetv2_dynamic_range.tflite"
tflite_ok = False
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # Dynamic-range quantization typically reduces stored weight size while keeping float input/output.
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    with open(TFLITE_PATH, "wb") as f:
        f.write(tflite_bytes)
    tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)
    tflite_ok = True
    print(f"TensorFlow Lite export succeeded: {tflite_size_mb:.2f} MB")
except Exception as exc:
    print("TensorFlow Lite conversion did not complete in this runtime.")
    print("Your trained Keras model is still available at:", KERAS_PATH)
    print("Conversion error:", repr(exc)[:800])


## 12. Summary table and save your model/results

In [ ]:
import pandas as pd

results = {
    "Test accuracy (%)": round(float(test_accuracy * 100), 2),
    "Test macro-F1": round(float(macro_f1), 4),
    "Keras model size (MB)": round(float(keras_size_mb), 2),
    "CPU latency per patch (ms)": round(float(cpu_latency_ms), 2),
    "Number of patches": int(len(X)),
    "Training patches": int(len(X_train)),
    "Validation patches": int(len(X_val)),
    "Test patches": int(len(X_test)),
    "TensorFlow Lite export": "Success" if tflite_ok else "Not completed"
}
results_df = pd.DataFrame([results])
display(results_df)
results_df.to_csv("/content/roverlite_results.csv", index=False)
print("Saved Keras model:", KERAS_PATH)
print("Saved results CSV: /content/roverlite_results.csv")
if tflite_ok:
    print("Saved TFLite model:", TFLITE_PATH)


## 13. Download the trained files

Run this final cell after model training and evaluation. It will download the Keras model, results CSV, and TensorFlow Lite model if conversion succeeded.

In [ ]:
from google.colab import files
files.download(KERAS_PATH)
files.download("/content/roverlite_results.csv")
if tflite_ok:
    files.download(TFLITE_PATH)


## How to explain this in your viva

- **Input:** a labelled rover-camera terrain patch from AI4Mars.
- **Ground truth:** the dominant terrain class in the matching segmentation mask.
- **Model:** MobileNetV2 transfer learning; the pretrained backbone extracts image features and a small Dense/Softmax head predicts four classes.
- **Edge AI aspect:** the trained model predicts locally on CPU and we measure model storage and single-patch latency. TensorFlow Lite export is an optional deployment step.
- **Caution rule:** predicted Sand or confidence below 0.60 displays an advisory caution. This is a simple demonstration rule and not a validated rover safety system.
- **Limitations:** labels can be incomplete or imbalanced; patch-level majority labels simplify complex scenes; Colab CPU benchmarking does not prove performance on a Raspberry Pi or an actual rover.

### References
1. NASA Open Data, [AI4Mars: A Dataset for Terrain-Aware Autonomous Driving on Mars](https://data.nasa.gov/dataset/ai4mars-a-dataset-for-terrain-aware-autonomous-driving-on-mars).
2. Hugging Face, [AI4MARS dataset mirror](https://huggingface.co/datasets/hassanjbara/AI4MARS).
3. NASA/JPL, [SPOC terrain classification](https://nasa-jpl.github.io/SPOC/) and [SPOC-Lite overview](https://software.nasa.gov/software/NPO-50647-1).